In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix, classification_report

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.tree import (DecisionTreeClassifier as DTC,
                          DecisionTreeRegressor as DTR,
                          plot_tree,
                          export_text)
from sklearn.metrics import (accuracy_score,
                             log_loss)
from sklearn.ensemble import \
     (RandomForestRegressor as RF,
      GradientBoostingRegressor as GBR)


In [10]:
df = pd.read_csv('Window_10_No_Overlap.csv', index_col=0, parse_dates=True)
df.head()
# Replace 'malignant' with 1 and 'benign' with 0
df = df.query("condition != 'black'")
# Multiple categories become 1
df['condition'] = df['condition'].replace({
    'neutral': 0,
    'positive': 1,
    'negative': 1
})
print(df['condition'].value_counts(dropna=False))

condition
1    276
0    138
Name: count, dtype: int64


/tmp/ipython-input-474/2437883669.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv('Window_10_No_Overlap.csv', index_col=0, parse_dates=True)
/tmp/ipython-input-474/2437883669.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['condition'] = df['condition'].replace({


In [12]:
#Do not do log, do z score for all variables
from sklearn.preprocessing import StandardScaler

# 1. Get a list of all columns EXCEPT 'condition'
cols_to_scale = df.columns.difference(['condition', 'source_rows'])

# 2. Initialize and apply the scaler only to those columns
scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

# Now df has standardized features, but 'condition' remains 0/1 (or its original state)

In [13]:
# 1. Define which columns are features (X)
# This excludes 'condition' so the model can't "cheat"
cols_to_scale = df.columns.difference(['condition', 'source_rows'])
X = df[cols_to_scale]

# 2. Define your target (y)
# Fixed the typo from 'condtion' to 'condition'
y = df['condition']

In [19]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix, classification_report
import timeit
start_time = timeit.default_timer()

condition_col = 'condition'  # rename the variable to avoid collision
colnames = df.drop(columns=[condition_col, 'source_rows']).columns.tolist()

formula = f'{condition_col} ~ {" + ".join(colnames)}'
model = smf.glm(formula=formula, data=df, family=sm.families.Binomial())

result = model.fit()
print(result.summary())

# Print computational time and accuracy
elapsed = timeit.default_timer() - start_time
print("--- Run time is %s seconds ---" % elapsed)

                 Generalized Linear Model Regression Results                  
Dep. Variable:              condition   No. Observations:                  368
Model:                            GLM   Df Residuals:                      327
Model Family:                Binomial   Df Model:                           40
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -201.42
Date:                Thu, 26 Feb 2026   Deviance:                       402.85
Time:                        01:17:30   Pearson chi2:                     364.
No. Iterations:                     6   Pseudo R-squ. (CS):             0.1371
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9151      0.133      6.856      0.0

In [23]:
import lightgbm as lgb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import timeit

# Define start time to calculate computation time
start_time = timeit.default_timer()

y_train = df['condition']
Y = df['condition']

# Initialize the LightGBM model
model = lgb.LGBMClassifier(learning_rate=0.09, max_depth=-5, random_state=42)

# Train the model
model.fit(X.values, y_train.values, eval_set=[(X.values, Y.values)], eval_metric='logloss')

# Predict on the test set
y_pred = model.predict(X.values)

# Calculate accuracy
accuracy = accuracy_score(Y, y_pred)


# Print computational time and accuracy
elapsed = timeit.default_timer() - start_time
print("--- Run time is %s seconds ---" % elapsed)
print(f"Accuracy on the test set: {accuracy:.2f}")

print(classification_report(Y, y_pred, digits=3))


[LightGBM] [Info] Number of positive: 276, number of negative: 138
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000347 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4191
[LightGBM] [Info] Number of data points in the train set: 414, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.666667 -> initscore=0.693147
[LightGBM] [Info] Start training from score 0.693147
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [24]:
from sklearn.model_selection import train_test_split
import timeit
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report

# Define start time to calculate computation time
start_time = timeit.default_timer()

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

# Initialize the LightGBM model
# Keeping the same parameters as before for consistency
model_split = lgb.LGBMClassifier(learning_rate=0.09, max_depth=-5, random_state=42)

# Train the model on the training set
model_split.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric='logloss')

# Predict on the test set
y_pred_split = model_split.predict(X_test)

# Calculate accuracy on the test set
accuracy_split = accuracy_score(y_test, y_pred_split)

# Print computational time and accuracy
elapsed_split = timeit.default_timer() - start_time
print("--- Run time for train-test split model is %s seconds ---" % elapsed_split)
print(f"Accuracy on the test set (with split): {accuracy_split:.2f}")

# Print classification report for the test set
print(classification_report(y_test, y_pred_split, digits=3))


[LightGBM] [Info] Number of positive: 221, number of negative: 110
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000338 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3500
[LightGBM] [Info] Number of data points in the train set: 331, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.667674 -> initscore=0.697682
[LightGBM] [Info] Start training from score 0.697682
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

In [25]:
import pandas as pd

feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_split.feature_importances_
}).sort_values('importance', ascending=False)

print(feat_imp)

   feature  importance
34     V40         112
16     V24          86
30     V37          54
3      V12          52
31     V38          49
18     V26          48
22      V3          47
21     V29          46
4      V13          42
0       V1          39
24     V31          39
19     V27          38
11      V2          33
29     V36          32
38      V8          32
26     V33          31
39      V9          30
25     V32          30
32     V39          29
35      V5          28
36      V6          27
33      V4          27
7      V16          26
27     V34          26
28     V35          26
20     V28          24
2      V11          24
1      V10          23
37      V7          23
5      V14          22
9      V18          22
15     V23          21
8      V17          20
14     V22          17
6      V15          15
10     V19          15
13     V21          12
12     V20          11
23     V30          10
17     V25           6
